# AquaCorrupt – feature extraction on a Colab GPU

This is the *one* step worth a GPU. It clones the repo, downloads the dataset, extracts frozen features for each backbone, and packages the cached embeddings so you can pull them back to your machine. Everything downstream (probe + plots) runs on CPU in seconds.

**Runtime > Change runtime type > GPU** before running.

In [ ]:
!nvidia-smi -L  # confirm a GPU is attached

In [ ]:
# 1) Clone your repo (replace with your GitHub URL after you push it)
REPO_URL = "https://github.com/<YOUR_USERNAME>/aquacorrupt.git"
!git clone $REPO_URL
%cd aquacorrupt
!pip -q install -r requirements.txt

In [ ]:
# 2) Kaggle creds -> download the MVP dataset.
#    Upload your kaggle.json (Kaggle > Settings > Create New Token) when prompted.
from google.colab import files
import os
up = files.upload()  # pick kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!python scripts/00_download_data.py --download

If the sanity check prints class folders under a different path than `data/corals/train` and `data/corals/valid`, open `config.py` and set `TRAIN_DIR` / `TEST_DIR` to the folders it found, then re-run the cell below.

In [ ]:
# 3) Extract frozen features on GPU (the whole point of Colab)
!python scripts/01_extract_features.py --device cuda

In [ ]:
# 4a) OPTION A: download the cached embeddings to your machine
!cd embeddings && zip -q -r ../embeddings.zip . && cd ..
from google.colab import files
files.download('embeddings.zip')  # then unzip into ./embeddings locally

In [ ]:
# 4b) OPTION B: run the probe + plot right here (they're cheap) and download the figure
!python scripts/02_run_probe.py
!python scripts/03_plot_curves.py
from google.colab import files
files.download('results/robustness_curve.png')